# MyStocks — Fase 3: ML Research

Dijalankan di Google Colab (bukan Codespaces), sesuai pembagian lingkungan riset vs build.

**Tujuan:** bandingkan baseline (Logistic Regression) vs XGBoost vs LightGBM untuk memprediksi
`P(harga naik >=5% dalam 10 hari trading sebelum stop-loss -2.5%)`, pakai walk-forward validation
yang time-aware (tanpa lookahead). **Notebook ini TIDAK memutuskan model final** — hanya
melaporkan hasil komparasi untuk direview sebelum model pemenang di-commit ke Codespaces.

Seluruh kode di bawah sudah diuji terhadap data nyata di Codespaces sebelum ditranskripsi ke sini
(termasuk 8 unit test untuk fungsi labeling, dan verifikasi matematis untuk walk-forward embargo)
— lihat `docs/BUILD_PROMPTS.md` Fase 3 untuk detail temuan/perbaikan selama pengujian.

**Cara pakai:**
1. Di Codespaces, jalankan `python -m scripts.export_for_colab` untuk membuat
   `data/export_for_colab_features.parquet` dan `..._prices.parquet`.
2. Upload kedua file itu ke Colab lewat cell "Upload data" di bawah.
3. Run All.

In [ ]:
!pip install -q "xgboost~=2.0.0" lightgbm shap pyarrow
# xgboost dipin ke 2.0.x (bukan versi terbaru): 2.1+ membundel nvidia-nccl (~300MB,
# tidak perlu untuk CPU-only) secara default, dan penting untuk konsistensi --
# model yang disimpan di sini akan dimuat oleh engine di Codespaces yang juga
# memakai xgboost 2.0.x (lihat requirements.txt).

## 1. Upload data
Upload `export_for_colab_features.parquet` dan `export_for_colab_prices.parquet` (dari `python -m scripts.export_for_colab` di Codespaces).

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih kedua file .parquet

In [ ]:
import pandas as pd

features = pd.read_parquet("export_for_colab_features.parquet")
prices = pd.read_parquet("export_for_colab_prices.parquet")
print(f"Loaded {len(features)} feature_daily rows, {len(prices)} price_history rows")

## 2. Triple-barrier labeling

`P(naik >=target_pct SEBELUM turun stop_pct, dalam `horizon` hari)`. Konvensi kalau kedua barrier
tersentuh di hari yang sama (ambigu dari OHLC harian): stop-loss menang (konservatif, tidak
melebih-lebihkan win rate). Butuh `horizon` hari penuh ke depan, kalau tidak cukup data -> NaN
(dikeluarkan dari training, bukan dipaksa 0/1) — diverifikasi dengan 8 test case tangan di Codespaces
sebelum dipakai di sini (termasuk kasus boundary: tepat di hari ke-10, tepat di luar horizon, dll).

In [ ]:
import numpy as np

HORIZON = 10
TARGET_PCT = 0.05
STOP_PCT = 0.025


def triple_barrier_label(high, low, close, entry_idx, horizon=10, target_pct=0.05, stop_pct=0.025):
    n = len(close)
    if entry_idx + horizon >= n:
        return np.nan
    entry_price = close[entry_idx]
    target_price = entry_price * (1 + target_pct)
    stop_price = entry_price * (1 - stop_pct)
    for i in range(entry_idx + 1, entry_idx + horizon + 1):
        if low[i] <= stop_price:
            return 0
        if high[i] >= target_price:
            return 1
    return np.nan


def build_panel_labels(prices: pd.DataFrame) -> pd.DataFrame:
    all_labels = []
    for code, g in prices.groupby("stock_code"):
        g = g.sort_values("date").reset_index(drop=True)
        high = g["high"].to_numpy(dtype=float)
        low = g["low"].to_numpy(dtype=float)
        close = g["close"].to_numpy(dtype=float)
        labels = [
            triple_barrier_label(high, low, close, i, HORIZON, TARGET_PCT, STOP_PCT)
            for i in range(len(g))
        ]
        all_labels.append(pd.DataFrame({"stock_code": code, "date": g["date"], "label": labels}))
    return pd.concat(all_labels, ignore_index=True)


labels = build_panel_labels(prices)
print("Label distribution:", labels["label"].value_counts(dropna=False).to_dict())

## 3. Walk-forward split (time-aware, dengan embargo)

Label hari-t bergantung pada `horizon` hari SETELAHNYA. Tanpa embargo, baris training dekat batas
fold bisa "mengintip" data yang seharusnya masuk test set. Embargo memastikan hanya baris yang
label-nya sudah sepenuhnya resolve SEBELUM batas train dipakai untuk training — diverifikasi
matematis di Codespaces: batas embargo persis pas (tidak bocor, tidak juga terlalu konservatif),
dikonfirmasi dengan menjalankan labeling function pada data terpotong vs data penuh dan hasilnya identik.

In [ ]:
def walk_forward_splits(dates, n_splits=5, test_size_days=100, min_train_days=600, label_horizon=10):
    unique_dates = np.sort(np.unique(dates))
    n = len(unique_dates)
    splits = []
    test_start_pos = min_train_days
    for _ in range(n_splits):
        if test_start_pos >= n:
            break
        test_end_pos = min(test_start_pos + test_size_days, n)
        embargo_end_pos = test_start_pos - 1 - label_horizon
        if embargo_end_pos < 0:
            test_start_pos = test_end_pos
            continue
        splits.append({
            "train_embargo_end_date": unique_dates[embargo_end_pos],
            "test_start_date": unique_dates[test_start_pos],
            "test_end_date": unique_dates[test_end_pos - 1],
        })
        test_start_pos = test_end_pos
    return splits

## 4. Siapkan panel feature + label

Catatan: `historical_win_rate` NaN pada ~44% baris (hari tanpa pola historis yang mirip) —
diisi netral 0.5 + ditambah flag `has_similar_pattern`, bukan drop baris (kalau di-drop, hampir
separuh data hilang cuma gara-gara satu kolom).

**Iterasi 2 (2026-08-24)**: iterasi pertama menyertakan kolom skala-absolut (`ema_9`, `sma_200`,
`macd_hist`, `obv`, dst) dan SHAP menunjukkan itu mendominasi feature importance — indikasi model
sebagian "menghafal saham" lewat level harga, bukan belajar pola general. Kolom-kolom itu
dikeluarkan di sini; model sekarang hanya memakai fitur yang sudah ternormalisasi (RSI, CMF, MFI,
ATR%, BB width%, jarak persentase ke MA/support/resistance, dst).

In [ ]:
NON_FEATURE_COLS = {"id", "stock_code", "date", "feature_version", "created_at"}
BOOL_COLS = ["higher_high_20d", "higher_low_20d", "lower_high_20d", "lower_low_20d"]

# Absolute price/volume-level columns -- NOT comparable across stocks with
# different price levels. Excluded per the iteration-1 finding above; rely on
# the already-normalized derivatives instead.
ABSOLUTE_SCALE_COLS = {
    "sma_20", "sma_50", "sma_200", "ema_9", "ema_20", "ema_50",
    "ema20_slope_5d", "ema20_accel_5d", "sma50_slope_10d",
    "macd", "macd_signal", "macd_hist", "macd_hist_slope_3d", "macd_hist_accel_3d",
    "volume_slope_5d", "obv", "obv_slope_5d",
}


def prepare_panel(features: pd.DataFrame, labels: pd.DataFrame):
    df = features.merge(labels, on=["stock_code", "date"], how="inner")
    df = df[df["sma_200"].notna()].copy()  # still used for the warmup filter, just not as a feature

    df["has_similar_pattern"] = (df["similar_pattern_count"].fillna(0) > 0).astype(int)
    df["historical_win_rate"] = df["historical_win_rate"].fillna(0.5)

    df = df[df["label"].notna()].copy()
    df["label"] = df["label"].astype(int)

    for c in BOOL_COLS:
        df[c] = df[c].astype(float)

    regime_dummies = pd.get_dummies(df["regime"], prefix="regime")
    df = pd.concat([df.drop(columns=["regime"]), regime_dummies], axis=1)

    feature_cols = [
        c for c in df.columns
        if c not in NON_FEATURE_COLS and c != "label" and c not in ABSOLUTE_SCALE_COLS
    ]

    remaining_na = df[feature_cols].isna().sum()
    remaining_na = remaining_na[remaining_na > 0]
    if len(remaining_na):
        print("WARNING: unexpected remaining NaN after preprocessing, dropping those rows:")
        print(remaining_na)
        df = df.dropna(subset=feature_cols)

    # feature_daily rows aren't guaranteed chronologically interleaved across
    # tickers -- sort explicitly, since equity-curve metrics (max drawdown)
    # depend on trade SEQUENCE, not just the set of returns.
    df = df.sort_values("date").reset_index(drop=True)
    return df, feature_cols


df, feature_cols = prepare_panel(features, labels)
print(f"Panel ready: {len(df)} rows, {len(feature_cols)} features")
print(f"Label balance: {df['label'].mean():.3f} positive rate")

## 5. Metrik evaluasi (ML + trading)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss


def trade_returns(y_true, y_pred_prob, threshold):
    taken = y_pred_prob > threshold
    outcomes = y_true[taken]
    return np.where(outcomes == 1, TARGET_PCT, -STOP_PCT)


def trading_metrics(returns):
    if len(returns) == 0:
        return {"n_trades": 0, "win_rate": np.nan, "total_return_pct": np.nan,
                "max_drawdown_pct": np.nan, "sharpe": np.nan, "profit_factor": np.nan}
    equity = np.cumprod(1 + returns)
    running_max = np.maximum.accumulate(equity)
    drawdown = (equity - running_max) / running_max
    wins = returns[returns > 0]
    losses = returns[returns < 0]
    profit_factor = wins.sum() / abs(losses.sum()) if len(losses) and losses.sum() != 0 else np.nan
    sharpe = (returns.mean() / returns.std() * np.sqrt(252 / HORIZON)) if returns.std() > 0 else np.nan
    return {
        "n_trades": len(returns),
        "win_rate": (returns > 0).mean(),
        "total_return_pct": (equity[-1] - 1) * 100,
        "max_drawdown_pct": drawdown.min() * 100,
        "sharpe": sharpe,
        "profit_factor": profit_factor,
    }


def ml_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob > threshold).astype(int)
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan,
        "brier": brier_score_loss(y_true, y_prob),
    }

## 6. Walk-forward: baseline vs XGBoost vs LightGBM

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

dates = df["date"].to_numpy()
splits = walk_forward_splits(dates, n_splits=5, test_size_days=100, min_train_days=600, label_horizon=HORIZON)
print(f"{len(splits)} walk-forward folds generated")

results = {"baseline": [], "xgboost": [], "lightgbm": []}
trading_results = {"baseline": [], "xgboost": [], "lightgbm": []}
last_fold_models = {}

for fold_i, split in enumerate(splits):
    train_mask = df["date"] <= split["train_embargo_end_date"]
    test_mask = (df["date"] >= split["test_start_date"]) & (df["date"] <= split["test_end_date"])

    X_train, y_train = df.loc[train_mask, feature_cols], df.loc[train_mask, "label"]
    X_test, y_test = df.loc[test_mask, feature_cols], df.loc[test_mask, "label"]

    if len(X_train) < 100 or len(X_test) < 20 or y_train.nunique() < 2:
        print(f"Fold {fold_i}: skipped (insufficient data or single-class train set)")
        continue

    print(f"Fold {fold_i}: train={len(X_train)} test={len(X_test)} "
          f"train_pos_rate={y_train.mean():.3f} test_pos_rate={y_test.mean():.3f}")

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    baseline = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
    prob_baseline = baseline.predict_proba(X_test_s)[:, 1]

    xgb_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, eval_metric="logloss", random_state=42)
    xgb_model.fit(X_train, y_train)
    prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

    lgb_model = lgb.LGBMClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, verbose=-1, random_state=42)
    lgb_model.fit(X_train, y_train)
    prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

    y_test_arr = y_test.to_numpy()
    for name, prob in [("baseline", prob_baseline), ("xgboost", prob_xgb), ("lightgbm", prob_lgb)]:
        results[name].append(ml_metrics(y_test_arr, prob))
        trading_results[name].append(trading_metrics(trade_returns(y_test_arr, prob, threshold=0.5)))

    last_fold_models = {"xgboost": xgb_model, "X_test": X_test}

## 7. Hasil komparasi (JANGAN pilih model final di sini — lapor & review dulu)

In [ ]:
print("="*70, "\nML METRICS (rata-rata across folds)\n", "="*70)
for name, fold_results in results.items():
    if not fold_results:
        continue
    print(f"\n{name}:")
    print(pd.DataFrame(fold_results).mean().to_string())

print("\n", "="*70)
print("TRADING METRICS (rata-rata across folds, threshold=0.5)")
print("CAVEAT: simplified serial backtest -- trades across ALL tickers compounded")
print("one after another in date order as if only one position is ever open at a")
print("time. Real trading holds multiple tickers concurrently, so total_return/")
print("max_drawdown here are illustrative, not a realistic portfolio simulation.")
print("Position sizing/concurrency is a Fase 4 (Risk Engine) concern.")
print("="*70)
for name, fold_results in trading_results.items():
    if not fold_results:
        continue
    print(f"\n{name}:")
    print(pd.DataFrame(fold_results).mean().to_string())

In [ ]:
import shap

if "xgboost" in last_fold_models:
    print("SHAP feature importance (fold terakhir, XGBoost, top 15)\n")
    explainer = shap.TreeExplainer(last_fold_models["xgboost"])
    shap_values = explainer.shap_values(last_fold_models["X_test"])
    importance = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols).sort_values(ascending=False)
    print(importance.head(15).to_string())

## 8. Temuan & rekomendasi

### Iterasi 1 (2026-08-24, 48 fitur termasuk skala-absolut)
- ROC-AUC ketiga model ~0.51-0.53. SHAP menunjukkan fitur skala-absolut (`ema_9`, `sma_200`,
  `obv`) mendominasi — indikasi model sebagian "menghafal saham" lewat level harga.
  XGBoost Sharpe **negatif** (-0.16), profit factor <1 (0.98) — secara trading tidak menarik.

### Iterasi 2 (2026-08-24, fitur skala-absolut dikeluarkan) — hipotesis terbukti benar
- SHAP sekarang didominasi fitur ternormalisasi: `atr_pct_14`, `distance_to_resistance_pct`,
  `bb_width_pct`, `mfi_slope_5d`, `similar_pattern_count`, `price_vs_sma50_pct` — jauh lebih
  masuk akal sebagai pola yang general lintas saham.
- ROC-AUC tetap serupa (~0.51-0.53) — wajar, AUC mengukur ranking di seluruh spektrum probability,
  sementara precision/recall di threshold tertentu dan metrik trading bisa membaik signifikan
  walau AUC agregat mirip.
- **Trading metrics membaik jelas** dibanding iterasi 1: XGBoost Sharpe naik dari -0.16 → **+0.86**,
  profit factor 0.98 → 1.45, max drawdown -50% → -28%. Baseline & LightGBM juga membaik
  (Sharpe masing-masing 0.71 dan 0.67). Jumlah trade menurun (model lebih selektif, precision naik
  dari ~0.35 → ~0.40-0.42) tapi kualitas tiap trade lebih baik.
- **XGBoost tampak paling menarik** di iterasi ini (Sharpe & profit factor tertinggi), tapi
  **TIDAK diputuskan sebagai model final di sini** — sesuai instruksi Fase 3, ini laporan
  komparasi untuk direview, bukan keputusan otomatis.

### Rekomendasi lanjutan (di luar scope Fase 3 saat ini)
- Dataset baru 10 ticker (blue-chip) x ~5 tahun. Menambah jumlah ticker yang di-track
  (`pipeline/tickers.py`) kemungkinan membantu generalisasi lebih dari sekadar tuning hyperparameter.
- CAVEAT metrik trading tetap berlaku: backtest serial (satu posisi pada satu waktu, bukan
  portofolio konkuren) — lihat catatan di bagian 7.

## 9. Export model produksi (XGBoost — dipilih user setelah review hasil di atas)

Dilatih ulang pada SELURUH data historis yang tersedia (bukan hanya satu fold walk-forward) —
walk-forward di atas sudah memberi sinyal validasi yang jujur; model produksi memakai semua data
supaya tidak membuang informasi terbaru. Disimpan lewat `get_booster().save_model()`, BUKAN
`model.save_model()` langsung — ditemukan bug nyata: wrapper sklearn `XGBClassifier.save_model()`
crash (`TypeError: _estimator_type undefined`) pada kombinasi xgboost 2.0.3 + scikit-learn 1.9,
sementara `get_booster().save_model()` + `xgb.Booster().load_model()` terverifikasi menghasilkan
prediksi identik.

In [ ]:
import json
import datetime as dt

final_model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, eval_metric="logloss", random_state=42)
final_model.fit(df[feature_cols], df["label"])

base_rate = float(df["label"].mean())

final_model.get_booster().save_model("direction_xgboost_v1.json")

metadata = {
    "model_version": "direction_xgboost_v1",
    "trained_at": dt.date.today().isoformat(),
    "feature_cols": feature_cols,
    "base_rate": base_rate,
    "horizon_days": HORIZON,
    "target_pct": TARGET_PCT,
    "stop_pct": STOP_PCT,
    "n_training_rows": len(df),
    "tickers": sorted(df["stock_code"].unique().tolist()) if "stock_code" in df.columns else None,
}
with open("direction_xgboost_v1_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved direction_xgboost_v1.json + metadata. Download both and commit to models/ in Codespaces:")
print(json.dumps(metadata, indent=2))

from google.colab import files
files.download("direction_xgboost_v1.json")
files.download("direction_xgboost_v1_metadata.json")